# Implementing Linear Regression

Line Equation:
y= intercept + (coefficient)*x

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import clear_output, display
import time
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

In [ ]:
class StandardScaler:
    def __init__(self):
        self.mean = None
        self.std = None
    def fit(self,X):
        X = np.array(X)
        self.mean = np.mean(X,axis=0)
        self.std = np.std(X,axis=0)
        self.std[self.std==0]=1
    def transform(self, X):
        X = np.array(X)
        return (X-self.mean)/self.std
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

In [ ]:
class Linear_Regression:
    def __init__(self,
        learning_rate=0.01,
        epochs=10000,
        batch_size=None,
        early_stopping=True,
        patience=10
    ):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.early_stopping = early_stopping
        self.patience = patience
        self.w=None
        self.b=0

        self.losses = []

        # self.mean = None
        # self.std = None
        self.scaler = StandardScaler()

    def _mse(self,y,y_pred):
        return np.mean((y-y_pred)**2)

    def r2_score(self,y,y_pred):
        ss_res = np.sum((y-y_pred)**2)
        ss_tot = np.sum((y-np.mean(y))**2)
        return 1-(ss_res/ss_tot)


    def fit(self, X, y,auto_plot=True,verbose=True):
        X = np.array(X)
        y=np.array(y)
        if X.ndim == 1:
            X = X.reshape(-1,1)
        # Store original X for visualization before scaling
        self.X_raw = X if X.ndim > 1 else X.reshape(-1, 1)
        self.y_raw = y
        X = self.scaler.fit_transform(X)
        n_samples, n_features = X.shape
        if len(y) != n_samples:
            raise ValueError("X and y must have the same number of samples")
        self.w=np.zeros(n_features)
        self.b=0
        batch_size = self.batch_size or n_samples
        best_loss = float("inf")
        patience_counter = 0
        for epoch in range(self.epochs):
            indices = np.random.permutation(n_samples)
            X_shuffled = X[indices]
            y_shuffled = y[indices]
            for i in range(0, n_samples, batch_size):
                X_batch = X_shuffled[i:i+batch_size]
                y_batch = y_shuffled[i:i+batch_size]
                y_pred = (X_batch @ self.w)+ self.b
                dw = (2/len(y_batch)) * (X_batch.T @ (y_pred-y_batch))
                db = (2/len(y_batch)) * (np.sum(y_pred-y_batch))

                self.w -= self.learning_rate*dw
                self.b -= self.learning_rate*db
            full_pred = X@self.w + self.b
            loss = self._mse(y,full_pred)
            self.losses.append(loss)
            if self.early_stopping:
                if loss<best_loss:
                    best_loss = loss
                    patience_counter=0
                else:
                    patience_counter+=1
                if patience_counter >= self.patience:
                    print(f'Early Stopping at epoch {epoch}')
                    break
        if verbose: print("Training Completed")
        # NEW: Automatically run the visualizations at the end!
        if auto_plot:
            self.plot_all_diagnostics(self.X_raw, self.y_raw)


    def fit_normal(self,X,y):
        X = np.array(X)
        y = np.array(y)
        if X.ndim == 1:
            X = X.reshape(-1,1)
        X = self.scaler.fit_transform(X)
        X_bias = np.c_[np.ones((X.shape[0],1)), X] # Adding a column of 1s to X to handle the bias (intercept) automatically
        # The Normal Equation: theta = (X^T * X)^-1 * X^T * y
        # Using pinv (pseudo-inverse) is safer than inv for singular matrices
        theta = np.linalg.pinv(X_bias.T @ X_bias) @ X_bias.T @ y

        self.b = theta[0]
        self.w = theta[1:]

    def predict(self,X):
        if self.w is None:
            raise ValueError("Model has not been trained yet")
        X=np.array(X)
        if X.ndim == 1:
            X = X.reshape(-1, self.w.shape[0])
        X = self.scaler.transform(X)
        return X@self.w+self.b

    def plot_all_diagnostics(self, X, y):
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        # Plot 1: Learning Curve
        axes[0].plot(self.losses, color='navy')
        axes[0].set_title('Learning Curve (Loss)')
        axes[0].set_xlabel('Epochs')
        axes[0].set_ylabel('MSE')
        axes[0].grid(True, alpha=0.3)

        # Plot 2: Actual vs Predicted
        y_pred = self.predict(X)
        axes[1].scatter(y, y_pred, alpha=0.5, color='teal')
        min_val, max_val = min(y.min(), y_pred.min()), max(y.max(), y_pred.max())
        axes[1].plot([min_val, max_val], [min_val, max_val], 'r--')
        axes[1].set_title(f'Actual vs Pred (R2: {self.r2_score(y, y_pred):.2f})')
        axes[1].set_xlabel('Actual')
        axes[1].set_ylabel('Predicted')

        # Plot 3: Residuals
        residuals = y - y_pred
        axes[2].scatter(y_pred, residuals, alpha=0.5, color='purple')
        axes[2].axhline(0, color='red', linestyle='--')
        axes[2].set_title('Residuals (Errors)')
        axes[2].set_xlabel('Predicted')

        plt.tight_layout()
        plt.show()

    def animate_training(self, X, y, frames=50, interval=100):
        """
        Generates a smooth video of the training process.
        frames: Number of snapshots to capture (e.g., 50 frames for the whole video).
        interval: Speed of animation in milliseconds (lower = faster).
        """
        X = np.array(X)
        y = np.array(y)
        if X.ndim == 1: X = X.reshape(-1, 1)

        if X.shape[1] > 1:
            print("Animation only supported for 1 feature datasets.")
            return

        # --- STEP 1: Run Training & Capture History ---
        # We store the state of the model at different timestamps
        history = []

        # Setup temporary model vars
        w = np.zeros(X.shape[1])
        b = 0
        X_scaled = self.scaler.fit_transform(X)
        n_samples = len(X)
        batch_size = self.batch_size or n_samples

        # Calculate how often to save a frame
        save_step = max(1, self.epochs // frames)

        for epoch in range(self.epochs):
            # Standard Gradient Descent Logic
            indices = np.random.permutation(n_samples)
            X_shuffled = X_scaled[indices]
            y_shuffled = y[indices]

            for i in range(0, n_samples, batch_size):
                X_batch = X_shuffled[i:i+batch_size]
                y_batch = y_shuffled[i:i+batch_size]
                y_pred = X_batch @ w + b
                dw = (2/len(y_batch)) * (X_batch.T @ (y_pred - y_batch))
                db = (2/len(y_batch)) * np.sum(y_pred - y_batch)
                w -= self.learning_rate * dw
                b -= self.learning_rate * db

            # Save snapshot for animation
            if epoch % save_step == 0:
                full_pred = X_scaled @ w + b
                loss = np.mean((y - full_pred)**2)
                # We copy() w so we don't just save a reference to the final w
                history.append((w.copy(), b, loss, epoch))

        # --- STEP 2: Create the Animation ---
        fig, ax = plt.subplots(figsize=(8, 6))

        # Plot Static Data
        ax.scatter(X, y, color='blue', alpha=0.5, label='Data')
        # Plot Initial Empty Line
        line, = ax.plot([], [], color='red', linewidth=3, label='Model Fit')

        ax.set_xlabel("Feature")
        ax.set_ylabel("Target")
        ax.legend()

        # Pre-calculate X range for smooth line plotting
        X_line_raw = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
        X_line_scaled = self.scaler.transform(X_line_raw)

        def update(frame_idx):
            # Retrieve history for this frame
            w_hist, b_hist, loss_hist, epoch_hist = history[frame_idx]

            # Calculate line position
            y_line_pred = X_line_scaled @ w_hist + b_hist

            # Update plot
            line.set_data(X_line_raw, y_line_pred)
            ax.set_title(f"Epoch: {epoch_hist} | Loss: {loss_hist:.4f}")
            return line,

        # Close the static plot so it doesn't show up separately
        plt.close()

        # Generate Animation
        print(f"Generating animation with {len(history)} frames...")
        anim = FuncAnimation(fig, update, frames=len(history), interval=interval, blit=True)

        # Return as HTML5 Video
        return HTML(anim.to_jshtml())

    def animate_training_static(self, X, y, frames=20, delay=0.01):

        """
        Runs a separate training loop purely for visualization.
        Only works for 1D input (1 Feature).
        """
        X = np.array(X)
        y = np.array(y)
        if X.ndim == 1: X = X.reshape(-1, 1)

        if X.shape[1] > 1:
            print("Animation only supported for 1 feature datasets.")
            return

        # 1. Setup Data & Scaler
        X_raw = X # Keep raw for plotting
        X_scaled = self.scaler.fit_transform(X)
        n_samples = len(X)

        # Reset weights just for this animation
        self.w = np.zeros(X.shape[1])
        self.b = 0
        self.losses = []
        batch_size = self.batch_size or n_samples

        # 2. Setup Plot
        fig, ax = plt.subplots(figsize=(8, 6))
        # Static scatter of data
        ax.scatter(X_raw, y, color='blue', alpha=0.5, label='Data')
        # Empty red line to be updated
        line, = ax.plot([], [], color='red', linewidth=3, label='Model')
        ax.legend()
        ax.set_xlabel("Feature")
        ax.set_ylabel("Target")

        # Pre-calc smooth X range for the red line (makes it look cleaner)
        X_line_raw = np.linspace(X_raw.min(), X_raw.max(), 100).reshape(-1, 1)
        X_line_scaled = self.scaler.transform(X_line_raw)

        # 3. Training Loop with Visual Updates
        update_interval = max(1, self.epochs // frames) # Don't plot every single epoch

        for epoch in range(self.epochs):
            # Gradient Descent Step
            indices = np.random.permutation(n_samples)
            X_shuffled = X_scaled[indices]
            y_shuffled = y[indices]

            for i in range(0, n_samples, batch_size):
                X_batch = X_shuffled[i:i+batch_size]
                y_batch = y_shuffled[i:i+batch_size]
                y_pred = X_batch @ self.w + self.b
                dw = (2/len(y_batch)) * (X_batch.T @ (y_pred - y_batch))
                db = (2/len(y_batch)) * np.sum(y_pred - y_batch)
                self.w -= self.learning_rate * dw
                self.b -= self.learning_rate * db

            # Calculate Loss
            full_pred = X_scaled @ self.w + self.b
            loss = self._mse(y, full_pred)
            self.losses.append(loss)

            # Update Plot (The Animation Part)
            if epoch % update_interval == 0:
                # Predict y values for the smooth line
                y_line_pred = X_line_scaled @ self.w + self.b

                # Update line position
                line.set_data(X_line_raw, y_line_pred)
                ax.set_title(f"Epoch: {epoch} | Loss: {loss:.4f}")

                # Refresh Display
                clear_output(wait=True)
                display(fig)
                time.sleep(delay)

        plt.close()
        print("Animation Complete.")


    # 1. Plot Loss (Did it learn?)
    def plot_loss(self):
        if not self.losses:
            print("No training history found. Train the model first.")
            return

        plt.figure(figsize=(10, 6))
        plt.plot(self.losses, color='navy', linewidth=2)
        plt.title('Training Loss over Epochs')
        plt.xlabel('Epochs')
        plt.ylabel('Mean Squared Error (MSE)')
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.show()

    # 2. Plot Residuals (Are there patterns in errors?)
    def plot_residuals(self, X, y):
        y_pred = self.predict(X)
        residuals = y - y_pred

        plt.figure(figsize=(10, 6))
        plt.scatter(y_pred, residuals, alpha=0.6, color='purple', edgecolor='w')
        plt.axhline(0, color='red', linestyle='--', linewidth=2)
        plt.title('Residual Plot (Diagnostics)')
        plt.xlabel('Predicted Values')
        plt.ylabel('Residuals (True - Pred)')
        plt.grid(True, alpha=0.3)
        plt.show()

    # 3. Plot Predictions (How accurate is it?)
    def plot_predictions(self, X, y):
        y_pred = self.predict(X)

        plt.figure(figsize=(8, 8))
        plt.scatter(y, y_pred, alpha=0.6, color='teal', edgecolor='w', label='Predictions')

        # Plot the "Perfect Fit" diagonal line
        min_val = min(np.min(y), np.min(y_pred))
        max_val = max(np.max(y), np.max(y_pred))
        plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2, label='Perfect Fit')

        r2 = self.r2_score(y, y_pred)
        plt.title(f'Actual vs Predicted (R2 Score: {r2:.3f})')
        plt.xlabel('Actual Values')
        plt.ylabel('Predicted Values')
        plt.legend()
        plt.grid(True)
        plt.show()

In [ ]:
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split

# 2. Create a simple 1D dataset (so Animation works)
# n_features=1 is crucial for the 2D line animation
X, y = make_regression(n_samples=200, n_features=1, noise=20, random_state=42)

# 3. Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Initialize your model
model = Linear_Regression(learning_rate=0.05, epochs=150)

# --- TEST A: Watch it learn (Animation) ---
print("Starting Animation...")
model.animate_training_static(X_train, y_train, frames=30)


# --- TEST B: Serious Training & Auto-Plot ---
print("\nStarting Official Training...")
# This should pop up 3 static graphs when finished
model.fit(X_train, y_train, auto_plot=True)

# 5. Check Test Accuracy
y_pred = model.predict(X_test)
print(f"Test R2 Score: {model.r2_score(y_test, y_pred):.4f}")

In [ ]:
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split

X, y = make_regression(n_samples=200, n_features=1, noise=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = Linear_Regression(learning_rate=0.05, epochs=150)

print("Starting Animation...")
display(model.animate_training(X_train, y_train, frames=30, interval=50))

In [ ]:
from sklearn.linear_model import LinearRegression as SklearnLR
from sklearn.metrics import mean_squared_error, r2_score

# 1. Create a complex dataset (Multiple features)
# This tests if your matrix multiplication works for higher dimensions
X, y = make_regression(n_samples=1000, n_features=25, noise=25, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- MODEL 1: Your Custom Implementation ---
print("--- Training Your Model ---")
my_model = Linear_Regression(learning_rate=0.01, epochs=2000, early_stopping=True)
my_model.fit(X_train, y_train, auto_plot=False, verbose=False) # Silence output for cleaner comparison

my_pred = my_model.predict(X_test)
my_mse = my_model._mse(y_test, my_pred)
my_r2 = my_model.r2_score(y_test, my_pred)

# --- MODEL 2: Scikit-Learn ---
print("--- Training Scikit-Learn ---")
sk_model = SklearnLR()
sk_model.fit(X_train, y_train)

sk_pred = sk_model.predict(X_test)
sk_mse = mean_squared_error(y_test, sk_pred)
sk_r2 = r2_score(y_test, sk_pred)

# --- COMPARISON RESULTS ---
print(f"\n{'Metric':<15} | {'Your Model':<15} | {'Sklearn':<15}")
print("-" * 50)
print(f"{'MSE (Lower=Better)':<15} | {my_mse:<15.4f} | {sk_mse:<15.4f}")
print(f"{'R2 (Higher=Better)':<15} | {my_r2:<15.4f} | {sk_r2:<15.4f}")

# Verdict
diff = abs(my_r2 - sk_r2)
if diff < 0.01:
    print("\n✅ SUCCESS: Your model matches Sklearn performance!")
else:
    print("\n⚠️ WARNING: Significant difference detected. Check learning rate or epochs.")

In [ ]:
# Create fake data to test
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split

X, y = make_regression(n_samples=500, n_features=3, noise=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# 1. Initialize & Train
model = Linear_Regression(learning_rate=0.01, epochs=500)
model.fit(X_train, y_train)
